In [1]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties

fun ensureFileExists(fileName: String): Boolean {
    val currentDir = System.getProperty("user.dir")
    val file = File(currentDir, fileName)

    if (!file.exists()) {
        file.createNewFile()
        return true
    }
    return false
}

private fun ensureFolderExists(fullPath: String) {
    val currentDir = System.getProperty("user.dir")
    val folder = File(fullPath)

    if (!folder.exists()) {
        folder.mkdirs()
    }
}

fun writeComparingCsv(headers: List<String>, instances: List<String>, revenues: List<List<Int>>, bestInstance: List<String>, fullPath: String) {
    ensureFolderExists(fullPath)

    csvWriter().open("$fullPath/comparison.csv") {
        writeRow(headers)
        instances.forEachIndexed{index, instance ->
            writeRow(listOf(instance) + revenues[index].map { it.toString() } + listOf(bestInstance[index]))
        }
    }

    println("CSV written successfully to $fullPath/comparison.csv")
}

fun <T : Any> writeCsv(data: List<T>, fileName: String, relativePath: String) {
    if (data.isEmpty()) {
        println("No data to write.")
        return
    }

    ensureFolderExists(relativePath)

    val kClass = data.first()::class
    val headers = kClass.declaredMemberProperties.map { it.name }

    csvWriter().open("$relativePath$fileName") {
        writeRow(headers)

        data.forEach { item ->
            val row = kClass.declaredMemberProperties.map { prop ->
                prop.getter.call(item)?.toString()?.replace(",", "\\,") ?: ""
            }
            writeRow(row)
        }
    }

    println("CSV written successfully to $relativePath$fileName")
}


fun compareTwoColumns (col1: DataColumn<*>, col2: DataColumn<*>, name: String): BaseColumn<String> {
    val col = col1.mapIndexed { index, value->
        val refValue = value as Int
        val compValue = col2[index] as Int
        if (refValue <= compValue) {
            (((compValue.toDouble() / refValue.toDouble()) - 1) * 100)
        }else {
            (((refValue.toDouble() / compValue.toDouble()) - 1) * -100)
        }.toInt().toString() + "\\%"
    }
    return col.rename(name)
}

private fun findDifferingSubstring(strings: List<String>): List<String> {
    val split = strings.map { it.split("_") }
    val transposed = split[0].indices.map { i -> split.map { it[i] } }

    val differingIndices = transposed
        .mapIndexedNotNull { index, parts ->
            if (parts.distinct().size > 1) index else null
        }

    return split.map { parts ->
        differingIndices.joinToString("_") { parts[it] }
    }
}
inline fun <reified T> transpose(xs: List<List<T>>): List<List<T>> {
    val cols = xs[0].size
    val rows = xs.size
    return List(cols) { j ->
        List(rows) { i ->
            xs[i][j]
        }
    }
}

fun mergeResults(fullpath: String) {
    val folder = File(fullpath)
    val files =  folder.listFiles()
        ?.filter { it.isFile && it.name != "comparison.csv"}
        ?: emptyList()
    val fileNames = findDifferingSubstring(files.map { it.nameWithoutExtension })

    val results = files.mapIndexed { index, file ->
        fileNames[index] to DataFrame.read(file)
    }
    val instances = results.first().second["name"].map { it.toString() }

    val bestAvgValues = instances.mapIndexed { index, _ ->
        val avgValues = results.map { it.second["revenueAvg"][index] as Int }
        val best = avgValues.maxOrNull()
        val bestIndex = avgValues.indexOf(best)
        results[bestIndex].first
    }
    val avgColumns = results.map {it.first to it.second["revenueAvg"] }

    writeComparingCsv(
        headers = listOf("instance") + fileNames + listOf("best"),
        instances = instances.toList(),
        revenues = transpose(avgColumns.map { it.second.toList().map { value -> value as Int} }),
        bestInstance = bestAvgValues.toList(),
        fullPath = fullpath
    )
}


In [2]:
// static stuff
enum class Context {CLUSTER, BUDGET, CLUSTERKM, CLUSTERALL, ELIMINATION}
enum class Mode {FLAT, RANDOM}
val percentageFraction = 2

val orderedInstances = listOf("eil101", "gil262", "pr299", "lin318", "rd400", "d493", "u574", "u724", "pcb1173", "fl1400", "pr2392").map { if (it == "instance") it else it + "-gen3-50" }

val baseline = "1.00"
val mode = Mode.FLAT
val context = Context.ELIMINATION
val bf = (baseline.toDouble() * 100) .toInt().toString()
val onlyTabular = true
// ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


enum class BaselineMode {INTABLE, EXTERNALTABLE}

val baselineMode = BaselineMode.EXTERNALTABLE
val baselinePath = "/op-solver-strict/results/ref/${mode.name.lowercase()}_1.csv"


val colsWithoutPercentages = when (context) {
    Context.CLUSTER -> baseline
    Context.BUDGET -> baseline
    Context.CLUSTERKM -> "kmn"
    Context.CLUSTERALL-> baseline
    Context.ELIMINATION -> ""
}

val colsToIgnoreWhenCalcuateMax = when (context) {
    Context.CLUSTER -> emptyList()
    Context.BUDGET -> emptyList()
    Context.CLUSTERKM -> emptyList()
    Context.CLUSTERALL -> listOf("kmn", "kmd")
    Context.ELIMINATION -> emptyList()
}

val relativePath = when (context) {
    Context.ELIMINATION -> "/op-solver-strict/results/elimination"
    Context.CLUSTERALL -> "/op-solver-strict/results/cluster/all${mode.name.lowercase()}"
    else -> "/op-solver-strict/results/${context.name.lowercase()}${mode.name.lowercase()}"
}
val header = when (context) {
    Context.CLUSTER -> listOf("instance", "rkmn", "rkmd", "nckmn", "nckmd", "fbckmn", "fbckmd", "best")
    Context.CLUSTERKM -> listOf("instance", "kmn", "kmd", "nckmn", "nckmd", "best")
    Context.CLUSTERALL -> listOf("instance", "rkmn", "rkmd", "nckmn", "nckmd", "fbckmn", "fbckmd", "kmn", "kmd", "best")
    Context.ELIMINATION -> listOf("instance","TSPrfb","TSPrce","OP")
    else -> null
}
data class OneRun(
    val budget: Int,
    val budgetSpentAvg: Double,
    val name: String,
    val revenueAvg: Int,
    val revenueMax: Int,
    val revenueMin: Int,
    val size: Int,
    val successfulAmount: Int,
    val timeAvg: Double,
    val timeMax: Double,
    val timeMin: Double
)

data class ClusterAll(
    val instance: String,
    val rkmn: Int,
    val rkmd: Int,
    val nckmn: Int,
    val nckmd: Int,
    val fbckmn: Int,
    val fbckmd: Int,
    val kmn: Int,
    val kmd: Int,
    val best: String
)

val label = "${context.toString().lowercase()}_${mode.toString().lowercase()}"

val caption = when (context) {
    Context.CLUSTER -> "Comparison of the clustering results for the instances in the ${mode.name.lowercase()} mode."
    Context.BUDGET -> "Comparison of the budget results for the instances in the ${mode.name.lowercase()} mode."
    Context.CLUSTERKM -> "Comparison of the k-means impact results for the instances in the ${mode.name.lowercase()} mode."
    Context.CLUSTERALL -> "Clustering Algorithm Comparison for ${mode.name.lowercase()} Instances. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at 90\\% of the maximum revenue. The star refers to the best mean revenue (kmn and kmd excluded)."
    Context.ELIMINATION -> "Comparison of the elimination methods for ${mode.name.lowercase()} the instances. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at 90\\% of the maximum revenue. The star refers to the best mean revenue."
}

val title = when (context) {
    Context.CLUSTER -> "clustering"
    Context.BUDGET -> "budget"
    Context.CLUSTERKM -> "k-means impact"
    Context.CLUSTERALL -> "clustering"
    Context.ELIMINATION -> "elimination comparison"
} + " ${mode.name.lowercase()}"

In [3]:
val pathToBaseLine  = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + baselinePath

var baselineDf = DataFrame.readCsv(pathToBaseLine)
baselineDf = baselineDf.sortWith (compareBy { row -> orderedInstances.indexOf(row["name"].toString()) })
    .reorderColumnsBy { colums -> header.indexOf(colums.name()) }
val colsToConvert = baselineDf.columnNames().drop(1)

baselineDf = colsToConvert.fold(baselineDf) { acc, col ->
    acc.convert(col) { v ->
        when (v) {
            is Number -> v.toInt()
            else -> v?.toString()?.trim()?.toInt()
                ?: throw IllegalArgumentException("Cannot parse column \$col value '\$v' to Int")
        }
    }
}
baselineDf


name,1.00,0.70,0.50,0.30
eil101,59,43,32,19
gil262,135,99,70,41
pr299,149,105,77,47
lin318,182,131,95,59
rd400,196,146,105,65
d493,304,204,147,66
u574,306,219,157,92
u724,382,278,196,110
pcb1173,574,409,293,176
fl1400,933,648,544,383


In [4]:
val pathToFolder  = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
.toString() + relativePath

val mainPath = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + relativePath + "/comparison_${mode.name.lowercase()}_$bf.csv"

if (!File(mainPath).exists()) {
  //  mergeResults(pathToFolder)
}

var df = DataFrame.readCsv(mainPath).cast<ClusterAll>()
df = df.sortWith (compareBy { row -> orderedInstances.indexOf(row["name"].toString()) })
    .reorderColumnsBy { colums -> header.indexOf(colums.name()) }
df


name,tsprfb,tsprce,op
eil101,59.000000,59.000000,59.000000
gil262,142.000000,137.000000,137.000000
pr299,148.000000,149.000000,150.000000
lin318,182.000000,179.000000,183.000000
rd400,196.000000,200.000000,192.000000
d493,303.000000,305.000000,302.000000
u574,310.000000,310.000000,309.000000
u724,380.000000,367.000000,384.000000
pcb1173,549.000000,545.000000,574.000000
fl1400,1039.000000,1037.000000,984.000000


In [6]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return if (onlyTabular) {
        """
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
         """
    } else {
        """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
    }
}

In [7]:
df = df.remove("best")
df.update("name").with { it.toString().split("-")[0] }
val amountColumns = (df.columns().size * 2 - 1).toString()

val formating = "|p{1.3cm}|" + List(df.columns().size-1) { "p{1.5cm} p{1.1cm}" }.joinToString("|") + "|"

val header = df.columnNames().first() + " & " +  df.columnNames()
    .filterIndexed{index, _ -> index != 0}.joinToString(separator = " & & ") + " &"

formating

|p{1.3cm}|p{1.5cm} p{1.1cm}|p{1.5cm} p{1.1cm}|p{1.5cm} p{1.1cm}|

In [8]:
val bestValues = df.convert { all()}.perRowCol { row, col ->
    if (col[row] is String || colsToIgnoreWhenCalcuateMax.contains(col.name())) {
        0
    } else {
        (col[row] as Number).toInt()
    }
}.map{ row ->
    row.rowMaxOf<Int>()
}

bestValues
baselineDf[baseline][1]
df

name,tsprfb,tsprce,op
eil101,59.000000,59.000000,59.000000
gil262,142.000000,137.000000,137.000000
pr299,148.000000,149.000000,150.000000
lin318,182.000000,179.000000,183.000000
rd400,196.000000,200.000000,192.000000
d493,303.000000,305.000000,302.000000
u574,310.000000,310.000000,309.000000
u724,380.000000,367.000000,384.000000
pcb1173,549.000000,545.000000,574.000000
fl1400,1039.000000,1037.000000,984.000000


In [9]:


val rowMaxValues = bestValues.mapIndexed { index, resultMax -> resultMax as Int }
//df.select { colsOf<Double>() }[1].rowMinOf<Double>()

In [10]:

val rowMinValues = df.map { row ->
    row.rowMinOf<Double>()
}

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df.select { colsOf<Double>() }[index]
        .rowMinOf<Double>()
    (maxEntry - minEntry).toDouble() / maxEntry.toDouble()
}
val maxRevenueDif = revenueDif.max()
val gradient = 0.1 //max(maxRevenueDif,0.0)

fun getSaturation (gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100-((maxValue - value) / (maxValue * gradient) * 100)),0.0),100.0).toInt().toString()
}

rowMaxValues


[59, 142, 150, 183, 200, 305, 310, 384, 574, 1039, 1212]

In [11]:
import java.util.Locale

fun calculatePercentage(refValue: Int, compValue: Int): Double{ return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())}
fun formatePercentage(value: Double): String { return "${String.format(Locale.US, "%.${percentageFraction}f", value * 100)}\\%" }

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = baselineDf.get(baseline)[row] as Int
    val percentage = calculatePercentage(refValue, compValue)
    return "${formatePercentage(percentage)}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if ((col.name() == "name") || col[row] is String) {
        10000.0
    } else  {
    val refValue = baselineDf[baseline][row] as Int
    calculatePercentage(refValue, (col[row] as Number).toInt())
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double) {
        formatePercentage(it) + "&"
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, 0.52\%&, 0.16\%&, 0.75\%&]

In [12]:
val stringdf = df.convert { all() }.perRowCol { row, col  ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = (col[row] as Number).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value && !colsToIgnoreWhenCalcuateMax.contains(col.name())){
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        }else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = (col[row] as Number).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value && !colsToIgnoreWhenCalcuateMax.contains(col.name())) {
            "\\cellcolor{cyan!$saturation} {$percentage}" +
                    "&\\cellcolor{cyan!$saturation}{\\textbf{$value*}}"
        }else {
            "\\cellcolor{cyan!$saturation} $percentage" +
                    "&\\cellcolor{cyan!$saturation}{$value}"
        }
    }
}
stringdf

name,tsprfb,tsprce,op
eil101,\cellcolor{cyan!100} {0.00\%}&\cellco...,\cellcolor{cyan!100} {0.00\%}&\cellco...,\cellcolor{cyan!100} {0.00\%}&\cellco...
gil262,\cellcolor{cyan!100} {5.19\%}&\cellco...,\cellcolor{cyan!64} 1.48\%&\cellcolor...,\cellcolor{cyan!64} 1.48\%&\cellcolor...
pr299,\cellcolor{cyan!86} -0.67\%&\cellcolo...,\cellcolor{cyan!93} 0.00\%&\cellcolor...,\cellcolor{cyan!100} {0.67\%}&\cellco...
lin318,\cellcolor{cyan!94} 0.00\%&\cellcolor...,\cellcolor{cyan!78} -1.65\%&\cellcolo...,\cellcolor{cyan!100} {0.55\%}&\cellco...
rd400,\cellcolor{cyan!80} 0.00\%&\cellcolor...,\cellcolor{cyan!100} {2.04\%}&\cellco...,\cellcolor{cyan!60} -2.04\%&\cellcolo...
d493,\cellcolor{cyan!93} -0.33\%&\cellcolo...,\cellcolor{cyan!100} {0.33\%}&\cellco...,\cellcolor{cyan!90} -0.66\%&\cellcolo...
u574,\cellcolor{cyan!100} {1.31\%}&\cellco...,\cellcolor{cyan!100} {1.31\%}&\cellco...,\cellcolor{cyan!96} 0.98\%&\cellcolor...
u724,\cellcolor{cyan!89} -0.52\%&\cellcolo...,\cellcolor{cyan!55} -3.93\%&\cellcolo...,\cellcolor{cyan!100} {0.52\%}&\cellco...
pcb1173,\cellcolor{cyan!56} -4.36\%&\cellcolo...,\cellcolor{cyan!49} -5.05\%&\cellcolo...,\cellcolor{cyan!100} {0.00\%}&\cellco...
fl1400,\cellcolor{cyan!100} {11.36\%}&\cellc...,\cellcolor{cyan!98} 11.15\%&\cellcolo...,\cellcolor{cyan!47} 5.47\%&\cellcolor...


In [13]:
val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ")  {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ")  {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


            \begin{tabular}{ |p{1.3cm}|p{1.5cm} p{1.1cm}|p{1.5cm} p{1.1cm}|p{1.5cm} p{1.1cm}|  }
                \hline
                \multicolumn{7}{|c|}{elimination comparison flat} \\
                \hline
                    name & tsprfb & & tsprce & & op & \\
                \hline
                    eil101 & \cellcolor{cyan!100} {0.00\%}&\cellcolor{cyan!100}{\textbf{59*}} & \cellcolor{cyan!100} {0.00\%}&\cellcolor{cyan!100}{\textbf{59*}} & \cellcolor{cyan!100} {0.00\%}&\cellcolor{cyan!100}{\textbf{59*}} \\ 
gil262 & \cellcolor{cyan!100} {5.19\%}&\cellcolor{cyan!100}{\textbf{142*}} & \cellcolor{cyan!64} 1.48\%&\cellcolor{cyan!64}{137} & \cellcolor{cyan!64} 1.48\%&\cellcolor{cyan!64}{137} \\ 
pr299 & \cellcolor{cyan!86} -0.67\%&\cellcolor{cyan!86}{148} & \cellcolor{cyan!93} 0.00\%&\cellcolor{cyan!93}{149} & \cellcolor{cyan!100} {0.67\%}&\cellcolor{cyan!100}{\textbf{150*}} \\ 
lin318 & \cellcolor{cyan!94} 0.00\%&\cellcolor{cyan!94}{182} & \cellcolor{cyan!78} -1.65\%&\cellcolor